In [ ]:
import wget
import os
os.chdir('/home/ga00693/2024-HL-ThermCL/notebooks/data/')

url1 = 'https://sol.spacenvironment.net/JB2008/indices/SOLFSMY.TXT'
url2 = 'https://sol.spacenvironment.net/JB2008/indices/DTCFILE.TXT'
url3 = 'https://www.celestrak.com/SpaceData/SW-All.csv'
wget_out_set_1 = wget.download(url1,'/home/ga00693/2024-HL-ThermCL/notebooks/data/')
print(f'downloaded at {wget_out_set_1}')
wget_out_set_2 = wget.download(url2,'/home/ga00693/2024-HL-ThermCL/notebooks/data/')
print(f'downloaded at {wget_out_set_2}')
wget_out_celestrack = wget.download(url3,'/home/ga00693/2024-HL-ThermCL/notebooks/data/')
print(f'downloaded at {wget_out_celestrack}')


In [ ]:
from ftplib import FTP
import datetime
import os
from glob import glob
import zipfile

def is_directory(ftp, item):
    """ Check if an item is a directory on the FTP server. """
    original_cwd = ftp.pwd()
    try:
        ftp.cwd(item)
        ftp.cwd(original_cwd)
        return True
    except Exception:
        return False
    
# Function to download files and directories recursively
# Function to download files and directories recursively

# Function to download files and directories recursively
def download_directory(ftp, path, local_dir):
    try:
        os.makedirs(local_dir, exist_ok=True)  # Ensure local directory exists
        ftp.cwd(path)
        print(f"Changed directory to {path}")
    except OSError:
        pass

    files = ftp.nlst()

    for file in files:
        local_path = os.path.join(local_dir, file)
        if is_directory(ftp, file):
            download_directory(ftp, file, local_path)
        else:
            if os.path.exists(local_path):
                print(f"File already exists: {local_path}, skipping download.                       ",end="\r")
            else:
                try:
                    with open(local_path, 'wb') as f:
                        ftp.retrbinary(f"RETR {file}", f.write)
                        print(f"Downloaded file: {file}                       ",end="\r")
                except ftplib.error_perm:
                    print(f"Cannot download: {file}                       ",end="\r")
    # Return to the parent directory on FTP and local file system
    ftp.cwd("..")
    os.chdir(os.path.dirname(local_dir))
start = datetime.datetime.now()
ftp = FTP('thermosphere.tudelft.nl')
ftp.login()
#passive mode
ftp.set_pasv(True)

# Directory to start downloading from (empty if the objective is to download everything)
starting_directory = ''

# Local directory to save files
local_base_dir = '/home/ga00693//2024-HL-ThermCL/notebooks/data/'

# Download all files and directories recursively from 'version_02'
download_directory(ftp, starting_directory, local_base_dir)

ftp.quit()

end = datetime.datetime.now()
diff = end - start
print(f'All files downloaded for {diff.seconds}s')

In [ ]:
import zipfile

def unzip_all_files(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.zip'):
                file_path = os.path.join(root, file)
                extract_dir = os.path.join(root, 'unzipped')
                
                # Check if the directory already exists
                if os.path.exists(os.path.join(extract_dir,file[:-4])):
                    print(f"Skipping {file_path} as it is already unzipped.")
                    continue
                
                with zipfile.ZipFile(file_path, 'r') as zip_ref:
                    zip_ref.extractall(extract_dir)
                print(f"Unzipped {file_path} in {extract_dir}")


In [ ]:
import pandas as pd
df=pd.read_csv('/home/ga00693/2024-HL-ThermCL/data/satellites_data.csv')

In [ ]:
df_set=pd.read_csv('/home/ga00693/2024-HL-ThermCL/data/set_sw.csv')

In [ ]:
import numpy as np
sw_data1 = np.loadtxt('/home/ga00693/2024-HL-ThermCL/scripts/data/SOLFSMY.TXT',usecols=range(2,11))
sw_data2 = np.loadtxt('/home/ga00693/2024-HL-ThermCL/scripts/data/DTCFILE.TXT',usecols=range(3,27),dtype=int)
swdata=(sw_data1,sw_data2)

In [ ]:
def get_sw_set(sw_data,t_mjd):
    '''
    Extract the necessary parameters describing the solar activity and geomagnetic activity from the space weather data.
    INPUT T1950 AND READ FILE FOR VALUES FOR F10, S10, M10, AND Y10
    READ ONE TIME AND STORE ALL FILE VALUES IN COMMON FOR RETRIEVAL
    Usage: 
    f107A,f107,ap,aph = get_sw(SW_OBS_PRE,t_ymd,hour)

    Inputs: 
    SW_OBS_PRE -> [2d str array] Content of the space weather data
    t_ymd -> [str array or list] ['year','month','day']
    hour -> []
    
    Outputs: 
    f107A -> [float] 81-day average of F10.7 flux
    f107 -> [float] daily F10.7 flux for previous day
    ap -> [int] daily magnetic index 
    aph -> [float array] 3-hour magnetic index 

    Examples:
    >>> f107A,f107,ap,aph = get_sw(SW_OBS_PRE,t_ymd,hour)
    '''
    sw_data1,sw_data2 = sw_data
    sw_mjd = sw_data1[:,0] - 2400000.5
    J_, = np.where(sw_mjd-0.5 < t_mjd)
    j = J_[-1]

    # USE 1 DAY LAG FOR F10 AND S10 FOR JB2008
    dlag = j-1
    F10,F10B,S10,S10B = sw_data1[dlag,1:5] 

    # USE 2 DAY LAG FOR M10 FOR JB2008
    dlag = j-2
    M10,M10B = sw_data1[dlag,5:7] 

    # USE 5 DAY LAG FOR Y10 FOR JB2008
    dlag = j-5
    Y10,Y10B = sw_data1[dlag,7:9] 
    
    t_dmjd = t_mjd - sw_mjd[j] + 0.5
    x = np.arange(0.5,48)/24
    y = sw_data2[j:j+2].flatten()
    DTCVAL = np.interp(t_dmjd,x,y)  
        
    return F10,F10B,S10,S10B,M10,M10B,Y10,Y10B,DTCVAL

In [ ]:
def jd_to_datetime(jd):    
    # Julian date of the Gregorian calendar epoch (January 1, 4713 BC)
    JD_GREGORIAN_EPOCH = 1721425.5
    
    # Calculate the number of days since the Gregorian calendar epoch
    days_since_epoch = jd - JD_GREGORIAN_EPOCH
    
    # Convert to datetime
    epoch = datetime.datetime(1, 1, 1)  # January 1, 1 AD
    date = epoch + datetime.timedelta(days=days_since_epoch)
    
    return date

In [ ]:
dict_set={"all__dates_datetime__":[],
          "space_environment_technologies__f107_obs__":[],
          "space_environment_technologies__f107_average__":[],
          "space_environment_technologies__s107_obs__":[],
          "space_environment_technologies__s107_average__":[],
          "space_environment_technologies__m107_obs__":[],
          "space_environment_technologies__m107_average__":[],
          "space_environment_technologies__y107_obs__":[],
          "space_environment_technologies__y107_average__":[],
          "JB08__d_st_dt__[K]":[]}
dates=[]
for i in range(len(sw_data1)):
    date=jd_to_datetime(sw_data1[i,0]).strftime('%Y-%m-%d')
    t_mjd=sw_data1[i,0] - 2400000.5
    f107_obs,f107_avg,s107_obs,s107_avg,m107_obs,m107_avg,y107_obs,y107_avg,dDst_dt=get_sw_set(swdata,t_mjd)
    dict_set['all__dates_datetime__'].append(date)
    dict_set['space_environment_technologies__f107_obs__'].append(f107_obs)
    dict_set['space_environment_technologies__f107_average__'].append(f107_avg)
    dict_set['space_environment_technologies__s107_obs__'].append(s107_obs)
    dict_set['space_environment_technologies__s107_average__'].append(s107_avg)
    dict_set['space_environment_technologies__m107_obs__'].append(m107_obs)
    dict_set['space_environment_technologies__m107_average__'].append(m107_avg)
    dict_set['space_environment_technologies__y107_obs__'].append(y107_obs)
    dict_set['space_environment_technologies__y107_average__'].append(y107_avg)
    dict_set['JB08__d_st_dt__[K]'].append(dDst_dt)
#save it as pandas csv
pd.DataFrame(dict_set).to_csv('/home/ga00693/2024-HL-ThermCL/data/set_sw.csv',index=False)

In [ ]:
#now the same for the celestrack one
sw_df = pd.read_csv('/home/ga00693/2024-HL-ThermCL/scripts/data/SW-All.csv')  
sw_df.dropna(subset=['C9'],inplace=True)
# Sort from newest date to past
sw_df.sort_values(by=['DATE'],ascending=False,inplace=True)
sw_df.reset_index(drop=True,inplace=True)

In [ ]:
def get_sw_celestrack(sw_df,t_ymd):
    j_, = np.where(sw_df['DATE'] == t_ymd)
    j = j_[0]
    f107A,f107,ap = sw_df.iloc[j]['F10.7_OBS_CENTER81'],sw_df.iloc[j+1]['F10.7_OBS'],sw_df.iloc[j]['AP_AVG']
    return f107A,f107,ap
dict_celestrack={"all__dates_datetime__" :[],
                 "celestrack__ap_average__":[]}
dates=sw_df['DATE'].values
for i in range(len(sw_df['DATE'].values)-1):    
    _,_,ap=get_sw_celestrack(sw_df,dates[i])
    dict_celestrack['all__dates_datetime__'].append(dates[i])
    dict_celestrack['celestrack__ap_average__'].append(ap)
#save it as pandas csv
df_celestrack=pd.DataFrame(dict_celestrack)
df_celestrack.sort_values(by=['all__dates_datetime__'],ascending=True,inplace=True)
df_celestrack.to_csv('/home/ga00693/2024-HL-ThermCL/data/celestrack_sw.csv',index=False)

In [ ]:
import pandas as pd
input_dir='/home/ga00693/2024-HL-ThermCL/data/'
df=pd.read_csv(input_dir+'satellites_data.csv')
dates=pd.DatetimeIndex(df['all__dates_datetime__'].values)
df['all__dates_datetime__']=dates

#celestrack sw data:
celestrack_sw=pd.read_csv(input_dir+'celestrack_sw.csv')
celestrack_sw['all__dates_datetime__']=pd.DatetimeIndex(celestrack_sw['all__dates_datetime__'])
celestrack_sw.sort_values(by='all__dates_datetime__', inplace=True)
#now the SET ones
set_sw=pd.read_csv(input_dir+'set_sw.csv')
set_sw['all__dates_datetime__']=pd.DatetimeIndex(set_sw['all__dates_datetime__'])
set_sw.sort_values(by='all__dates_datetime__', inplace=True)

sw_data = pd.merge_asof(celestrack_sw,
                              set_sw, 
                              on='all__dates_datetime__', 
                              direction='backward')

df_merged = pd.merge_asof(df,
                          sw_data, 
                          on='all__dates_datetime__', 
                          direction='backward')
del df

to_drop=[]
for col in df_merged.columns:
    if col.startswith('Unn'):
        to_drop.append(col)
if len(to_drop)>0:
    df_merged.drop(to_drop,axis=1,inplace=True)


In [ ]:
input_dir

In [ ]:
import numpy as np
    
df_merged.replace([np.inf, -np.inf], np.nan,inplace=True)
df_merged.dropna(axis=0,inplace=True)
df_merged.reset_index(drop=True,inplace=True)
df_merged.sort_values(by='all__dates_datetime__', inplace=True)
df_merged.reset_index(drop=True,inplace=True)
#now let's also add msise density data as an extra column:
df_merged.to_csv(input_dir+'satellites_data_w_sw.csv',index=False)

df_nrlmsise00=pd.read_csv('densities_msise.csv')
df_merged['NRLMSISE00__thermospheric_density__[kg/m**3]']=df_nrlmsise00.values.flatten()
df_merged.reset_index(drop=True,inplace=True)

In [ ]:
import pandas as pd
import numpy as np
import os

input_dir='/home/ga00693/2024-HL-ThermCL/data/'
df=pd.read_csv(os.path.join(input_dir,'satellites_data.csv'))
dates=pd.DatetimeIndex(df['all__dates_datetime__'].values)
df['all__dates_datetime__']=dates

#celestrack sw data:
celestrack_sw=pd.read_csv(os.path.join(input_dir,'celestrack_sw.csv'))
celestrack_sw['all__dates_datetime__']=pd.DatetimeIndex(celestrack_sw['all__dates_datetime__'])
celestrack_sw.sort_values(by='all__dates_datetime__', inplace=True)
#now the SET ones
set_sw=pd.read_csv(os.path.join(input_dir,'set_sw.csv'))
set_sw['all__dates_datetime__']=pd.DatetimeIndex(set_sw['all__dates_datetime__'])
set_sw.sort_values(by='all__dates_datetime__', inplace=True)

sw_data = pd.merge_asof(celestrack_sw,
                              set_sw, 
                              on='all__dates_datetime__', 
                              direction='backward')

df_merged = pd.merge_asof(df,
                          sw_data, 
                          on='all__dates_datetime__', 
                          direction='backward')
del df

to_drop=[]
for col in df_merged.columns:
    if col.startswith('Unn'):
        to_drop.append(col)
if len(to_drop)>0:
    df_merged.drop(to_drop,axis=1,inplace=True)
    
df_merged.replace([np.inf, -np.inf], np.nan,inplace=True)
df_merged.dropna(axis=0,inplace=True)
df_merged.reset_index(drop=True,inplace=True)
df_merged.sort_values(by='all__dates_datetime__', inplace=True)
df_merged.reset_index(drop=True,inplace=True)
#now let's also add msise density data as an extra column:
df_nrlmsise00=pd.read_csv('densities_msise.csv')
df_merged['NRLMSISE00__thermospheric_density__[kg/m**3]']=df_nrlmsise00.values.flatten()
df_merged.reset_index(drop=True,inplace=True)
df_merged.to_csv('satellites_data_w_sw.csv',index=False)

#subset of 1% of the data (taken randomly):
sampled_values = df_merged.sample(frac=0.01, random_state=1)
sampled_values.sort_values(by='all__dates_datetime__', inplace=True)
sampled_values.reset_index(drop=True,inplace=True)
sampled_values.to_csv('satellites_data_w_sw_2mln.csv',index=False)

#description csv (useful for holding statistics about the data)
df_describe=df_merged.describe()
df_describe.to_csv('satellites_data_w_sw_describe.csv',index=False)



In [ ]:
df.columns

In [ ]:
dict_celestrack={"all__dates_datetime__" :[],
                 "celestrack__ap_average__":[]}
dates=sw_df['DATE'].values
for i in range(len(sw_df['DATE'].values)-1):    
    _,_,ap=get_sw_celestrack(sw_df,dates[i])
    dict_celestrack['all__dates_datetime__'].append(dates[i])
    dict_celestrack['celestrack__ap_average__'].append(ap)
#save it as pandas csv
df_celestrack=pd.DataFrame(dict_celestrack)
df_celestrack.sort_values(by=['all__dates_datetime__'],ascending=True,inplace=True)
df_celestrack.to_csv('/home/ga00693/2024-HL-ThermCL/data/celestrack_sw.csv',index=False)


In [ ]:
df_2

In [ ]:
sw_df

In [ ]:
df

In [ ]:
df_2=pd.read_csv('/home/ga00693/2024-HL-ThermCL/data/set_sw.csv')

In [ ]:
df_set

In [ ]:
df_2.iloc[6:]

In [ ]:
get_sw_set(swdata,sw_data1[0,0]-2400000.5)

In [ ]:
swd

In [ ]:
import datetime
datetime.datetime.strptime(f'{sw_data1[0,0]}', '%y%j').date()

In [ ]:
df_set

In [ ]:
import wget
url1 = 'https://sol.spacenvironment.net/JB2008/indices/SOLFSMY.TXT'
url2 = 'https://sol.spacenvironment.net/JB2008/indices/DTCFILE.TXT'
url3 = 'https://www.celestrak.com/SpaceData/SW-All.csv'
download_dir='/home/jupyter/data/'
wget_out_set_1 = wget.download(url1,download_dir)
print(f'downloaded at {wget_out_set_1}')
wget_out_set_2 = wget.download(url2,download_dir)
print(f'downloaded at {wget_out_set_2}')
wget_out_celestrack = wget.download(url3,download_dir)
print(f'downloaded at {wget_out_celestrack}')
